**Amrit Dhillon**
**GSB545 Homework #2 Boosting Model Notebook**

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

#data
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']

X_test = test.drop(columns=['id'])

#encode categorical target for XGBoost
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

#make categorical columns category dtype for XGBoost
cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

#CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#hyperparam grid for tuning
param_grid_xgb = {
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

xgb = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss'
)

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    scoring='balanced_accuracy',
    cv=skf,
    n_jobs=1
)

grid_xgb.fit(X, y_encoded)

print("Best XGBoost params:", grid_xgb.best_params_)
print("Best XGBoost CV score:", grid_xgb.best_score_)

Best XGBoost params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}
Best XGBoost CV score: 0.9620168768535653


In [4]:
best_xgb = grid_xgb.best_estimator_
best_xgb.fit(X, y_encoded)

xgb_preds_encoded = best_xgb.predict(test)
xgb_preds = label_encoder.inverse_transform(xgb_preds_encoded)

submission_xgb = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': xgb_preds
})

submission_xgb.to_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_xgb.csv', index=False)
submission_xgb.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [1]:
#XGBoost version 2 with better tuning and engineered features
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

#loading in data
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

#adding some engineered features
train['Moisture_to_Heat_Ratio'] = train['Soil_Moisture'] / (train['Temperature_C'] + 1)
test['Moisture_to_Heat_Ratio'] = test['Soil_Moisture'] / (test['Temperature_C'] + 1)

train['Water_Availability_Index'] = (train['Rainfall_mm'] + train['Previous_Irrigation_mm']) / (train['Field_Area_hectare'] + 1)
test['Water_Availability_Index'] = (test['Rainfall_mm'] + test['Previous_Irrigation_mm']) / (test['Field_Area_hectare'] + 1)

train['Climate_Stress_Index'] = (train['Temperature_C'] * train['Sunlight_Hours'] * (1 + train['Wind_Speed_kmh'] / 25)) / (train['Humidity'] + 1)
test['Climate_Stress_Index'] = (test['Temperature_C'] * test['Sunlight_Hours'] * (1 + test['Wind_Speed_kmh'] / 25)) / (test['Humidity'] + 1)

#splitting features and target
X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

#enconding target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

#categorical columns to category dtype for XGBoost
cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

#sample weights for class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_encoded)

#stratified CV
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

#XGBoost model
xgb = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss',
    tree_method='hist'
)

#param grid for randomized search CV
param_dist_xgb = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [100, 150, 200],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

random_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=8,
    scoring='balanced_accuracy',
    cv=skf,
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_xgb.fit(X, y_encoded, sample_weight=sample_weights)

print("Best XGBoost params:", random_xgb.best_params_)
print("Best XGBoost CV score:", random_xgb.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost params: {'subsample': 1.0, 'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best XGBoost CV score: 0.9694184973068083


In [2]:
best_xgb = random_xgb.best_estimator_

best_xgb.fit(X, y_encoded, sample_weight=sample_weights)

xgb_preds_encoded = best_xgb.predict(X_test)
xgb_preds = label_encoder.inverse_transform(xgb_preds_encoded)

submission_xgb = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': xgb_preds
})

submission_xgb.to_csv(
    '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_xgb_improved.csv',
    index=False
)

submission_xgb.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [3]:
#just checking for submission
row_count = len(submission_xgb)
print(f'Total rows: {row_count}')

Total rows: 270000


In [ ]:
"""import os
import pandas as pd
from catboost import CatBoostClassifier

# load data
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

# split features / target
X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

# categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
cat_features_idx = [X.columns.get_loc(col) for col in cat_cols]

# make a writable CatBoost working directory
train_dir = '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/catboost_info'
os.makedirs(train_dir, exist_ok=True)

# pre-create common subfolders to avoid local dir issues
for sub in ['tmp', 'learn', 'test']:
    os.makedirs(os.path.join(train_dir, sub), exist_ok=True)

# small parameter grid
param_grid_cat = {
    'depth': [6, 8],
    'learning_rate': [0.05, 0.1],
    'iterations': [100, 200]
}

# model
cat_model = CatBoostClassifier(
    random_state=42,
    verbose=0,
    loss_function='MultiClass',
    cat_features=cat_features_idx,
    train_dir=train_dir
)

# native CatBoost grid search
grid_result = cat_model.grid_search(
    param_grid=param_grid_cat,
    X=X,
    y=y,
    cv=5,
    stratified=True,
    shuffle=True,
    verbose=False
)

print("Best CatBoost params:", grid_result["params"])"""

###catboost code I decided not to use for now, its taking too long to run, XGBoost worked fairly well for now

In [ ]:
"""cat_preds = cat_model.predict(X_test).reshape(-1)

submission_cat = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': cat_preds
})

submission_cat.to_csv(
    '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_catboost.csv',
    index=False
)

submission_cat.head()""""
#more CatBoost code that I don't need for now but might use later